# 02 — Leakage review: what is known before the call? (Step 1.6)

**Goal:** for each of the 19 input features other than `duration`, decide from *evidence* whether its value would be known
at the moment we make the prediction, and flag anything uncertain.

**Scope:** feature availability and data leakage only. This notebook does **not** train any model, create a train / development / stream
split, or change the data (it only reads `data/raw/bank-additional-full.csv`). Recommendations below are **provisional**: where the
documentation is silent, the decision is left pending for the team.

**Source of definitions:** `bank-additional-names.txt` from the official UCI Bank Marketing archive (the `bank-additional-full.csv` version,
41,188 rows, 20 inputs). The CSV in `data/raw/` has the same SHA-256 as the one inside that archive. Quotes below are copied verbatim.

## 1. What "data leakage" means, and which moment are we predicting at?

**Data leakage** = the model is trained on information that would not exist at the time it has to make its prediction
(like grading a student with an answer key written after the exam). It makes scores look great in the notebook and fail in real use.
Two flavours:
- **Hard leakage:** the feature is recorded after, or because of, the outcome.
- **Timing leakage:** the feature is real, but its value was not yet recorded or published when we would need it.

Whether a feature leaks depends on **when the prediction is made**. Two candidate moments:

| | **Moment A: before any contact is scheduled** | **Moment B: immediately before a planned call begins** *(confirmed 2026-10-27)* |
|---|---|---|
| Question answered | *Whom should we put on the call list?* | *Will this customer subscribe on the call that is about to start?* |
| Known | Customer attributes, history of earlier campaigns, market context at planning time | Everything in A, **plus** the planned channel and date, and how many times this customer was already called in this campaign |
| Not yet known | Channel, date, number of calls in this campaign | Call length (`duration`) and the outcome |

**Implications of choosing Moment B:** more features become *allowed* (`contact`, `month`, `day_of_week`, possibly `campaign`),
but only if the dataset's values equal the **planned/pre-call** values. The dataset documents them as properties of the *last contact*,
so B makes the meaning of a row (one call? one customer's whole campaign?) the key uncertainty. If we chose Moment A instead, those
four features would have to be excluded, and the model would answer a different (arguably more useful for targeting) question.

> **Confirmed 2026-10-27 and recorded in `DECISIONS.md`:** we use Moment B. It fixes the prediction time so that only information available before the call may be used; the current call's duration and outcome are not available.
>
> **Choosing Moment B does not automatically approve ambiguous features.** `contact`, `month`, `day_of_week`, `campaign` and the economic indicators are only *allowed in principle*; whether each was really known before the modeled call is still unresolved (see sections 4-7).

In [1]:
import pandas as pd
pd.set_option('display.max_colwidth', None)

df = pd.read_csv('../data/raw/bank-additional-full.csv', sep=';')   # read-only: nothing is written by this notebook
assert df.shape == (41188, 21)
inputs = [c for c in df.columns if c not in ('y', 'duration')]
print('Rows:', len(df), '| input features reviewed here (all inputs except duration):', len(inputs))
print(inputs)

Rows: 41188 | input features reviewed here (all inputs except duration): 19
['age', 'job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'month', 'day_of_week', 'campaign', 'pdays', 'previous', 'poutcome', 'emp.var.rate', 'cons.price.idx', 'cons.conf.idx', 'euribor3m', 'nr.employed']


## 2. What the UCI documentation says (verbatim)

In [2]:
doc_quotes = {
    'age':            'age (numeric)   [section: "bank client data"]',
    'job':            'job : type of job (categorical: "admin.","blue-collar",...,"unknown")   [section: "bank client data"]',
    'marital':        'marital : marital status (categorical: "divorced","married","single","unknown"; note: "divorced" means divorced or widowed)   [section: "bank client data"]',
    'education':      'education (categorical: "basic.4y",...,"university.degree","unknown")   [section: "bank client data"]',
    'default':        'default: has credit in default? (categorical: "no","yes","unknown")   [section: "bank client data"]',
    'housing':        'housing: has housing loan? (categorical: "no","yes","unknown")   [section: "bank client data"]',
    'loan':           'loan: has personal loan? (categorical: "no","yes","unknown")   [section: "bank client data"]',
    'contact':        'contact: contact communication type (categorical: "cellular","telephone")   [section: "related with the last contact of the current campaign"]',
    'month':          'month: last contact month of year (categorical: "jan", "feb", "mar", ..., "nov", "dec")   [same section]',
    'day_of_week':    'day_of_week: last contact day of the week (categorical: "mon","tue","wed","thu","fri")   [same section]',
    'campaign':       'campaign: number of contacts performed during this campaign and for this client (numeric, includes last contact)   [section: "other attributes"]',
    'pdays':          'pdays: number of days that passed by after the client was last contacted from a previous campaign (numeric; 999 means client was not previously contacted)   [section: "other attributes"]',
    'previous':       'previous: number of contacts performed before this campaign and for this client (numeric)   [section: "other attributes"]',
    'poutcome':       'poutcome: outcome of the previous marketing campaign (categorical: "failure","nonexistent","success")   [section: "other attributes"]',
    'emp.var.rate':   'emp.var.rate: employment variation rate - quarterly indicator (numeric)   [section: "social and economic context attributes"]',
    'cons.price.idx': 'cons.price.idx: consumer price index - monthly indicator (numeric)   [same section]',
    'cons.conf.idx':  'cons.conf.idx: consumer confidence index - monthly indicator (numeric)   [same section]',
    'euribor3m':      'euribor3m: euribor 3 month rate - daily indicator (numeric)   [same section]',
    'nr.employed':    'nr.employed: number of employees - quarterly indicator (numeric)   [same section]',
}
assert list(doc_quotes) == inputs, 'the quote table must cover exactly the 19 reviewed features, in file order'
pd.Series(doc_quotes, name='UCI documentation').to_frame()

,UCI documentation
age,"age (numeric) [section: ""bank client data""]"
job,"job : type of job (categorical: ""admin."",""blue-collar"",...,""unknown"") [section: ""bank client data""]"
marital,"marital : marital status (categorical: ""divorced"",""married"",""single"",""unknown""; note: ""divorced"" means divorced or widowed) [section: ""bank client data""]"
education,"education (categorical: ""basic.4y"",...,""university.degree"",""unknown"") [section: ""bank client data""]"
default,"default: has credit in default? (categorical: ""no"",""yes"",""unknown"") [section: ""bank client data""]"
housing,"housing: has housing loan? (categorical: ""no"",""yes"",""unknown"") [section: ""bank client data""]"
loan,"loan: has personal loan? (categorical: ""no"",""yes"",""unknown"") [section: ""bank client data""]"
contact,"contact: contact communication type (categorical: ""cellular"",""telephone"") [section: ""related with the last contact of the current campaign""]"
month,"month: last contact month of year (categorical: ""jan"", ""feb"", ""mar"", ..., ""nov"", ""dec"") [same section]"
day_of_week,"day_of_week: last contact day of the week (categorical: ""mon"",""tue"",""wed"",""thu"",""fri"") [same section]"


Other relevant sentences in the same file:
- *"bank-additional-full.csv with all examples, ordered by date (from May 2008 to November 2010)."*
- On the five economic features: *"national wide indicators from a ~10M population country, published by the Banco de Portugal"*. **The file says nothing about publication dates or lags.**
- On `duration`: *"the duration is not known before a call is performed ... should be discarded if the intention is to have a realistic predictive model."* (already excluded)
- Nothing in the file says whether one row is one phone call or one customer's whole campaign, and there is no customer ID column.

## 3. Check A — are `pdays`, `previous` and `poutcome` purely historical?

Documentation: all three describe **earlier** campaigns. We check that the three columns agree with one another.

In [3]:
print(pd.crosstab([df.pdays.eq(999).rename('pdays == 999'), df.previous.eq(0).rename('previous == 0')], df.poutcome))
mismatch = (df.pdays == 999) & (df.previous > 0)
print('\nRows with pdays == 999 ("not previously contacted") but previous > 0 (contacted before):', int(mismatch.sum()))
print('Their poutcome values:', df.loc[mismatch, 'poutcome'].value_counts().to_dict())
print('Rows with pdays != 999 but previous == 0:', int(((df.pdays != 999) & (df.previous == 0)).sum()))

poutcome                    failure  nonexistent  success
pdays == 999 previous == 0                               
False        False              142            0     1373
True         False             4110            0        0
             True                 0        35563        0

Rows with pdays == 999 ("not previously contacted") but previous > 0 (contacted before): 4110
Their poutcome values: {'failure': 4110}
Rows with pdays != 999 but previous == 0: 0


**Reading it:** all three are defined in terms of *previous* campaigns, which is consistent with being known before any call in the current campaign.
No row has a recorded previous *success* without a previous contact. One **inconsistency with the documentation** shows up, though: for the rows above, `previous > 0`
says the customer *was* contacted before, while `pdays = 999` says they were *not*. This is an encoding question (how to treat `pdays = 999`, already a pending decision), not evidence of leakage.
The documentation does not explain it.

## 4. Check B — what does one row of `campaign` record?

Documentation: *"number of contacts performed during this campaign and for this client (numeric, includes last contact)"*.
Two readings are possible, and they have opposite consequences under Moment B:

1. **Row = one call; `campaign` = that call's sequence number** (1st, 2nd, ... call to this customer). The number is known when the call is planned, so no leakage.
2. **Row = one customer's campaign; `campaign` = the final total** of calls. The total includes calls made *after* the moment we predict, and who gets called again can depend on the outcome (e.g. calling stops after a "yes"). That would be leakage.

Having "includes last contact" does **not** by itself decide between them. With no customer ID we cannot follow one customer across rows, so we use only a *necessary condition*:
if every row were the Nth call of a **complete** call log, then there could never be more rows with `campaign = k` than with `campaign = k-1` (a 37th call needs a 36th call row).

In [4]:
print('min campaign:', int(df.campaign.min()), '| max campaign:', int(df.campaign.max()))
counts = df.campaign.value_counts().reindex(range(1, int(df.campaign.max()) + 1), fill_value=0)
violations = pd.DataFrame({'k': counts.index[1:], 'rows_with_campaign_k_minus_1': counts.values[:-1], 'rows_with_campaign_k': counts.values[1:]})
violations = violations[violations.rows_with_campaign_k > violations.rows_with_campaign_k_minus_1]
print('Values of k where count(k) > count(k-1):', len(violations))
violations.reset_index(drop=True)

min campaign: 1 | max campaign: 56
Values of k where count(k) > count(k-1): 10


,k,rows_with_campaign_k_minus_1,rows_with_campaign_k
0,17,51,58
1,20,26,30
2,27,8,11
3,29,8,10
4,35,3,5
5,37,0,1
6,39,0,1
7,40,1,2
8,42,1,2
9,56,0,1


In [5]:
# Descriptive only: yes rate by campaign value (capped at 10). A pattern here is NOT proof of leakage or of its absence.
by_campaign = df.groupby(df.campaign.clip(upper=10)).y.agg(rows='size', yes_rate=lambda s: round((s == 'yes').mean(), 3))
by_campaign.index.name = 'campaign (10 = 10 or more)'
by_campaign

,rows,yes_rate
campaign (10 = 10 or more),,
1,17642,0.130
2,10570,0.115
3,5341,0.107
4,2651,0.094
5,1599,0.075
6,979,0.077
7,629,0.060
8,400,0.042
9,283,0.060


**Reading it:** the necessary condition is **violated** (for example a row with `campaign = 37` but none with 36, and one with 56 but none with 55). So the rows are *not* a complete log of every call.
That argues against reading 1 *in its simplest form*, but it does not prove reading 2: some calls could simply be missing from the public file
(the documentation says only that some attributes were removed for privacy; it does not say whether any rows were removed). The falling yes rate with more contacts is consistent with either reading and is not evidence either way.

An extra test I tried outside this notebook, matching rows that share the same customer attributes, gave results no different from shuffled labels, so it was uninformative and is not included here.

**Conclusion:** the documentation and the data **cannot settle** what `campaign` records. This stays **uncertain**.

## 5. Check C — `contact`, `month`, `day_of_week`

Documentation: all three sit under *"related with the last contact of the current campaign"*; `month` and `day_of_week` are the *last contact* month and weekday.
Under Moment B a planned call's channel, month and weekday are normally known in advance **if** the row describes that call. They would be
established only afterwards if the row describes a *later final* contact (reading 2 above) or if the channel used differs from the planned one.
Nothing in the data distinguishes these cases, so the verdict here is tied to the unresolved row-meaning question in Check B. A descriptive look:

In [6]:
print(df.contact.value_counts().to_dict())
print(df.month.value_counts().to_dict())
print(df.day_of_week.value_counts().to_dict())

{'cellular': 26144, 'telephone': 15044}
{'may': 13769, 'jul': 7174, 'aug': 6178, 'jun': 5318, 'nov': 4101, 'apr': 2632, 'oct': 718, 'sep': 570, 'mar': 546, 'dec': 182}
{'thu': 8623, 'mon': 8514, 'wed': 8134, 'tue': 8090, 'fri': 7827}


Separately from leakage: because rows are ordered by date, `month` also carries *where in the timeline* a row sits, so it interacts with drift. That is a modelling concern for later steps, not a leakage finding.

## 6. Check D — the five economic indicators: measurement frequency vs. availability

Documented *frequency*: `emp.var.rate` and `nr.employed` quarterly, `cons.price.idx` and `cons.conf.idx` monthly, `euribor3m` daily. Frequency says how often a series
**is measured**, not when each value **became public** (publication usually lags the period it measures). The documentation gives no release dates.
What the data can show is *when the value attached to the rows changes*. Runs below follow the file's row order and use the `month` column only (no reconstructed years).

In [7]:
econ = ['emp.var.rate', 'cons.price.idx', 'cons.conf.idx', 'euribor3m', 'nr.employed']
print('Distinct values:', {c: int(df[c].nunique()) for c in econ})

def value_runs(col):
    """Consecutive rows (file order) that share the same value of `col`."""
    run_id = (df[col] != df[col].shift()).cumsum()
    return df.groupby(run_id).agg(value=(col, 'first'), rows=(col, 'size'), first_month=('month', 'first'), last_month=('month', 'last')).reset_index(drop=True)

print('Number of runs of identical consecutive values:', {c: len(value_runs(c)) for c in econ})
value_runs('nr.employed')

Distinct values: {'emp.var.rate': 10, 'cons.price.idx': 26, 'cons.conf.idx': 26, 'euribor3m': 316, 'nr.employed': 11}
Number of runs of identical consecutive values: {'emp.var.rate': 11, 'cons.price.idx': 26, 'cons.conf.idx': 26, 'euribor3m': 434, 'nr.employed': 11}


,value,rows,first_month,last_month
0,5191.0,7763,may,may
1,5228.1,16234,jun,aug
2,5195.8,3683,oct,nov
3,5176.3,10,dec,dec
4,5099.1,8534,mar,may
5,5076.2,1663,jun,aug
6,5017.5,1071,sep,nov
7,5023.5,172,dec,dec
8,5008.7,650,mar,may
9,4991.6,773,jun,aug


In [8]:
value_runs('emp.var.rate')

,value,rows,first_month,last_month
0,1.1,7763,may,may
1,1.4,16234,jun,aug
2,-0.1,3683,oct,nov
3,-0.2,10,dec,dec
4,-1.8,8534,mar,may
5,-2.9,1663,jun,aug
6,-3.4,1071,sep,nov
7,-3.0,172,dec,dec
8,-1.8,650,mar,may
9,-1.7,773,jun,aug


In [9]:
print('cons.price.idx: runs that span more than one month name:',
      int((value_runs('cons.price.idx').first_month != value_runs('cons.price.idx').last_month).sum()),
      'of', len(value_runs('cons.price.idx')))
print('cons.conf.idx : runs that span more than one month name:',
      int((value_runs('cons.conf.idx').first_month != value_runs('cons.conf.idx').last_month).sum()),
      'of', len(value_runs('cons.conf.idx')))

cons.price.idx: runs that span more than one month name: 0 of 26


cons.conf.idx : runs that span more than one month name: 0 of 26


**Reading it:**
- The two *quarterly* series have only 10-11 distinct values and change value at the start of **Mar, Jun, Sep and Dec**, not at calendar-quarter starts (Jan, Apr, Jul, Oct). So the value attached to, say, June-August rows is not simply "the figure for the calendar quarter these rows fall in".
  This is *consistent with* a lag (the most recently available figure being used), but it does **not prove** the value was public on the call date, and the documentation does not say how the values were matched to calls.
- The two *monthly* series change at month boundaries; `euribor3m` has 316 distinct values (consistent with a daily series).
- **Conclusion:** the documentation does not tell us *when* each value was published. Checking this would need an external source (the Banco de Portugal release calendar), which we have not consulted. This stays **uncertain** for all five.
- Whether to use economic features at all is a *separate*, still-pending decision (Step 3.6). This notebook only asks whether they were *known in time*.

## 7. Feature groups and availability assumptions (Moment B)

Moment B is confirmed, but it only defines *when* we predict. Each group still needs its own availability assumption.

| Group | Columns | Status under Moment B |
|---|---|---|
| 1. Customer attributes | `age`, `job`, `marital`, `education`, `default`, `housing`, `loan` | **Provisionally available**, assuming the customer record reflects information already captured before the call. |
| 2. Previous campaign history | `pdays`, `previous`, `poutcome` | **Provisionally available**, subject to verifying their historical meaning. The 4,110 rows with `previous > 0` but `pdays = 999` remain a **documented interpretation issue** (Check A). |
| 3. Current contact information | `campaign`, `contact`, `month`, `day_of_week` | **Unresolved** until we establish whether the recorded values were known before the modeled call (Checks B and C). |
| 4. Economic indicators | `emp.var.rate`, `cons.price.idx`, `cons.conf.idx`, `euribor3m`, `nr.employed` | **Unresolved**: the documentation gives measurement frequency but not all publication dates and delays (Check D). |
| 5. Excluded | `duration` | **Excluded**: it becomes known only after the call. |
| 6. Analysis metadata | `row_order`, reconstructed `period` | **Not predictive inputs.** They exist only in `data/processed/bank_marketing_for_modeling.csv` (created by notebook 01), for ordering and labelling. The reconstructed year-month labels remain a pending decision. |

> The dictionary below is a **review artifact for this notebook**, not the project's settings. The shared feature lists will live in the Step 2.1 settings module in `src/`.

In [10]:
availability_groups = {
    '1 Customer attributes':          ['age', 'job', 'marital', 'education', 'default', 'housing', 'loan'],
    '2 Previous campaign history':    ['pdays', 'previous', 'poutcome'],
    '3 Current contact information':  ['campaign', 'contact', 'month', 'day_of_week'],
    '4 Economic indicators':          ['emp.var.rate', 'cons.price.idx', 'cons.conf.idx', 'euribor3m', 'nr.employed'],
}
not_inputs = {'5 Excluded (known only after the call)': ['duration'],
              '6 Analysis metadata, not predictive inputs': ['row_order', 'period']}

grouped = [col for cols in availability_groups.values() for col in cols]
assert sorted(grouped) == sorted(inputs), 'groups 1-4 must cover exactly the 19 reviewed inputs'
assert len(grouped) == len(set(grouped)), 'a feature must not appear in two groups'
assert not set(grouped) & {col for cols in not_inputs.values() for col in cols}
group_of = {col: g for g, cols in availability_groups.items() for col in cols}
print({g: len(cols) for g, cols in availability_groups.items()}, '| total:', len(grouped))
print('metadata columns present in the raw CSV?', [col for col in ('row_order', 'period') if col in df.columns])

{'1 Customer attributes': 7, '2 Previous campaign history': 3, '3 Current contact information': 4, '4 Economic indicators': 5} | total: 19
metadata columns present in the raw CSV? []


## 8. Provisional assessment of all 19 features

Moment B (confirmed) is the reference for the "known immediately before the planned call?" column. **No feature-inclusion decision is made here**; "Investigate" means the evidence we have is insufficient.

In [11]:
rows = [('age', 'Customer age', 'Listed under "bank client data". No timing stated.', 'Likely yes (bank-held customer attribute)', 'No evidence of leakage. Documentation does not say when the customer record was captured.', 'Include (provisional)'), ('job', 'Customer job type', 'Under "bank client data". No timing stated.', 'Likely yes', 'No evidence of leakage. Has "unknown" values (handling pending).', 'Include (provisional)'), ('marital', 'Marital status', 'Under "bank client data". No timing stated.', 'Likely yes', 'No evidence of leakage. Has "unknown" values.', 'Include (provisional)'), ('education', 'Education level', 'Under "bank client data". No timing stated.', 'Likely yes', 'No evidence of leakage. Has "unknown" values.', 'Include (provisional)'), ('default', 'Has credit in default?', 'Under "bank client data". No timing stated.', 'Likely yes', 'No evidence of leakage. 20.9% "unknown".', 'Include (provisional)'), ('housing', 'Has housing loan?', 'Under "bank client data". No timing stated.', 'Likely yes', 'No evidence of leakage. Has "unknown" values.', 'Include (provisional)'), ('loan', 'Has personal loan?', 'Under "bank client data". No timing stated.', 'Likely yes', 'No evidence of leakage. Has "unknown" values.', 'Include (provisional)'), ('contact', 'Channel (cellular / telephone) of the LAST contact of the current campaign', '"related with the last contact of the current campaign".', 'Uncertain: yes only if the row describes the planned call and the planned channel was the one used', 'Depends on the unresolved row meaning (Check B). Planned vs. actual channel is not documented.', 'Investigate further'), ('month', 'Month of the LAST contact', '"last contact month of year".', 'Uncertain: a planned call date is normally known in advance if the row describes that call', 'Same row-meaning uncertainty. Also acts as a timeline proxy (drift concern, not leakage).', 'Investigate further'), ('day_of_week', 'Weekday of the LAST contact', '"last contact day of the week".', 'Uncertain: same as month', 'Same row-meaning uncertainty.', 'Investigate further'), ('campaign', 'Number of contacts in this campaign for this client, including the last one', '"number of contacts performed during this campaign and for this client (numeric, includes last contact)".', 'Uncertain: yes if it is the sequence number of the call being planned; no if it is the final total', 'Check B: neither reading can be confirmed or ruled out. A complete call log is contradicted, a customer-level final total is not proven.', 'Investigate further'), ('pdays', 'Days since last contact in a PREVIOUS campaign (999 = never)', '"after the client was last contacted from a previous campaign"; "999 means client was not previously contacted".', 'Yes (refers to an earlier campaign)', 'Not leakage. 4,110 rows have previous > 0 but pdays = 999, contradicting the documentation (Check A); encoding is a pending decision.', 'Include (provisional); encoding pending'), ('previous', 'Number of contacts BEFORE this campaign', '"number of contacts performed before this campaign and for this client".', 'Yes (earlier campaigns)', 'Not leakage. See the pdays inconsistency in Check A.', 'Include (provisional)'), ('poutcome', 'Outcome of the PREVIOUS campaign', '"outcome of the previous marketing campaign".', 'Yes (earlier campaign)', 'Not leakage in the data: values agree with pdays/previous apart from the pdays = 999 inconsistency. Mostly "nonexistent".', 'Include (provisional)'), ('emp.var.rate', 'Employment variation rate (national)', '"quarterly indicator"; published by Banco de Portugal.', 'Uncertain: frequency is not publication time', 'Values change at Mar/Jun/Sep/Dec (Check D), consistent with a lag but unproven. Release dates are not in the documentation.', 'Investigate further (inclusion itself is a separate pending decision)'), ('cons.price.idx', 'Consumer price index (national)', '"monthly indicator".', 'Uncertain: frequency is not publication time', 'Changes at month boundaries. Publication lag unknown.', 'Investigate further (same note)'), ('cons.conf.idx', 'Consumer confidence index (national)', '"monthly indicator".', 'Uncertain: frequency is not publication time', 'Changes at month boundaries. Publication lag unknown.', 'Investigate further (same note)'), ('euribor3m', 'Euribor 3-month interest rate', '"daily indicator".', 'Plausibly yes, not verified from the documentation', 'Daily series (316 distinct values). Publication timing relative to the call time is not documented.', 'Investigate further (same note)'), ('nr.employed', 'Number of employees (national)', '"quarterly indicator".', 'Uncertain: frequency is not publication time', 'Same pattern as emp.var.rate.', 'Investigate further (same note)')]
assessment = pd.DataFrame(rows, columns=['feature', 'what it represents', 'UCI documentation evidence', 'known immediately before the planned call?', 'leakage / uncertainty', 'recommendation (provisional)']).set_index('feature')
assert list(assessment.index) == inputs
assessment.insert(0, 'group', assessment.index.map(group_of))
assessment

,group,what it represents,UCI documentation evidence,known immediately before the planned call?,leakage / uncertainty,recommendation (provisional)
feature,,,,,,
age,1 Customer attributes,Customer age,"Listed under ""bank client data"". No timing stated.",Likely yes (bank-held customer attribute),No evidence of leakage. Documentation does not say when the customer record was captured.,Include (provisional)
job,1 Customer attributes,Customer job type,"Under ""bank client data"". No timing stated.",Likely yes,"No evidence of leakage. Has ""unknown"" values (handling pending).",Include (provisional)
marital,1 Customer attributes,Marital status,"Under ""bank client data"". No timing stated.",Likely yes,"No evidence of leakage. Has ""unknown"" values.",Include (provisional)
education,1 Customer attributes,Education level,"Under ""bank client data"". No timing stated.",Likely yes,"No evidence of leakage. Has ""unknown"" values.",Include (provisional)
default,1 Customer attributes,Has credit in default?,"Under ""bank client data"". No timing stated.",Likely yes,"No evidence of leakage. 20.9% ""unknown"".",Include (provisional)
housing,1 Customer attributes,Has housing loan?,"Under ""bank client data"". No timing stated.",Likely yes,"No evidence of leakage. Has ""unknown"" values.",Include (provisional)
loan,1 Customer attributes,Has personal loan?,"Under ""bank client data"". No timing stated.",Likely yes,"No evidence of leakage. Has ""unknown"" values.",Include (provisional)
contact,3 Current contact information,Channel (cellular / telephone) of the LAST contact of the current campaign,"""related with the last contact of the current campaign"".",Uncertain: yes only if the row describes the planned call and the planned channel was the one used,Depends on the unresolved row meaning (Check B). Planned vs. actual channel is not documented.,Investigate further
month,3 Current contact information,Month of the LAST contact,"""last contact month of year"".",Uncertain: a planned call date is normally known in advance if the row describes that call,"Same row-meaning uncertainty. Also acts as a timeline proxy (drift concern, not leakage).",Investigate further


In [12]:
assessment['recommendation (provisional)'].str.extract(r'^(Include|Investigate)')[0].value_counts()

0
Include        10
Investigate     9
Name: count, dtype: int64

## 9. Modeling implications: a conservative vs. a broader first baseline

*No choice is made here and no model is trained. This only lays out the trade-off for the team lead's decision.*

| | **Conservative baseline** | **Broader baseline** |
|---|---|---|
| Features | Groups 1 and 2 (10 features) | Groups 1 and 2, plus group 3 (4 features) and group 4 (5 features) = 19 |
| Strength | Every input is plausibly known before the call, so the claim "no timing leakage" is easy to defend. | Uses all information the dataset offers; may predict better. |
| Risk | Possibly weaker predictions and an incomplete picture of what drives subscription. We have not measured how much weaker. | If any group-3 or group-4 value was *not* known in time, scores are inflated, and we could not tell from the scores alone. |
| Effect on the drift study | Economic indicators are absent, so macro drift cannot hurt the model directly; drift would have to show up in customer, history or contact features. | The economic indicators are the features that move most between early and late data (PSI up to 11.5 in notebook 01's *illustrative* 70/30 comparison), so retraining strategies would be tested against the strongest drift. |

Points to keep in mind:
- The dataset authors report that adding the economic features *"lead to substantial improvement in the prediction of a success, even when the duration of the call is not included"*. That is their claim, not a result we measured.
- These are not the only two options: groups 3 and 4 can be added separately (e.g. history plus economic, without the contact group).
- A with/without comparison (planned for Step 3.6) can show how much the results depend on the unresolved groups, but it cannot show *whether those features were known in time*. Only documentation or external sources can answer that.
- **The feature-set choice belongs to the team lead.** It is not made here, and nothing in this notebook should be read as an approved feature list for model comparison.

## 10. Summary, limits and open questions

- **10 features** show no sign of leakage and are provisionally fine: the 7 customer attributes (documentation silent on capture time) and `pdays`, `previous`, `poutcome` (historical by definition; only `pdays = 999` has an encoding inconsistency).
- **4 features are unresolved** (`contact`, `month`, `day_of_week`, `campaign`) because the documentation does not say whether a row is one call or one customer's whole campaign, and our checks could not settle it.
- **5 economic features are unresolved** for *timing*: the documentation gives measurement frequency but no publication dates.
- **Limits:** no customer ID, so no customer-level reconstruction; the original paper (Moro et al., 2014) was **not** retrieved, so it may answer the open questions; external release calendars were not consulted; descriptive tables show consistency, not causation.

**Confirmed:** Moment B, recorded in `DECISIONS.md` on 2026-10-27. It does not approve any ambiguous feature.

**Pending (the first two are logged in `DECISIONS.md`; the last two were already pending):**
1. Whether `campaign`, `contact`, `month` and `day_of_week` were known before the modeled call.
2. When each economic indicator's value was publicly available relative to call dates.
3. Whether economic indicators are included as model features, and which baseline (conservative, broader, or in between) comes first.
4. How to handle `pdays = 999` given the inconsistency found in Check A.